In [ ]:


import time
import pandas as pd
import torch
import torchvision.models as models

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# -----------------------------------------------------
# BENCHMARK FUNCTION
# -----------------------------------------------------

def benchmark_model(
    model,
    model_name,
    input_size=(1,3,224,224),
    runs=500
):

    model = model.to(device)
    model.eval()

    dummy = torch.randn(
        input_size
    ).to(device)

    # -------------------------
    # PARAMS
    # -------------------------

    params = sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6

    # -------------------------
    # WARMUP
    # -------------------------

    for _ in range(50):

        with torch.no_grad():

            _ = model(dummy)

    torch.cuda.synchronize()

    # -------------------------
    # TIMING
    # -------------------------

    starter = time.time()

    for _ in range(runs):

        with torch.no_grad():

            _ = model(dummy)

    torch.cuda.synchronize()

    ender = time.time()

    total_time = ender - starter

    time_per_image = (
        total_time / runs
    ) * 1000

    fps = (
        1000 / time_per_image
    )

    return [
        model_name,
        round(params,2),
        round(time_per_image,3),
        round(fps,2)
    ]

# =====================================================
# MODELS
# =====================================================

benchmark_results = []

# -----------------------------------------------------
# ResNet50
# -----------------------------------------------------

benchmark_results.append(

    benchmark_model(
        models.resnet50(
            weights=None
        ),
        "ResNet50"
    )

)

# -----------------------------------------------------
# DenseNet121
# -----------------------------------------------------

benchmark_results.append(

    benchmark_model(
        models.densenet121(
            weights=None
        ),
        "DenseNet121"
    )

)

# -----------------------------------------------------
# EfficientNetB0
# -----------------------------------------------------

benchmark_results.append(

    benchmark_model(
        models.efficientnet_b0(
            weights=None
        ),
        "EfficientNetB0"
    )

)

# -----------------------------------------------------
# MobileNetV3
# -----------------------------------------------------

benchmark_results.append(

    benchmark_model(
        models.mobilenet_v3_large(
            weights=None
        ),
        "MobileNetV3"
    )

)

# -----------------------------------------------------
# Proposed Encoder
# -----------------------------------------------------

benchmark_results.append([
    "Proposed (MobileNetV3+ECA)",
    4.76,
    ms_per_image,      # previous code output
    fps                # previous code output
])

# =====================================================
# TABLE
# =====================================================

benchmark_df = pd.DataFrame(

    benchmark_results,

    columns=[
        "Model",
        "Params(M)",
        "Time/Image(ms)",
        "FPS"
    ]
)

print("\n")
print("="*80)
print("MODEL COMPLEXITY ANALYSIS")
print("="*80)

display(
    benchmark_df
)

benchmark_df.to_csv(
    "model_complexity_analysis.csv",
    index=False
)

print(
    "\nSaved: model_complexity_analysis.csv"
)